In [1]:
%load_ext autoreload
%autoreload 2
import os
home_dir = os.path.expanduser('~')
os.chdir(os.path.join(home_dir, "DeepFact"))

In [2]:
# Cell 1: imports
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

from huggingface_hub import hf_hub_download

# Cell 2: config
repo_id = "kkkevinkkk/DeepFactBench"
repo_type = "dataset"

sentences_filename = "test.json"
reports_filename = "test_reports.json"

local_dir = Path("data/deep_fact_bench")
local_dir.mkdir(parents=True, exist_ok=True)

local_sentences_path = local_dir / "test.json"
local_reports_path = local_dir / "test_reports.json"
output_dir = local_dir / "test_reports_split"

# Cell 3: download both files from Hugging Face
downloaded_sentences_path = hf_hub_download(
    repo_id=repo_id,
    repo_type=repo_type,
    filename=sentences_filename,
)

downloaded_reports_path = hf_hub_download(
    repo_id=repo_id,
    repo_type=repo_type,
    filename=reports_filename,
)

print("Downloaded sentences:", downloaded_sentences_path)
print("Downloaded reports:", downloaded_reports_path)

test_reports.json: 0.00B [00:00, ?B/s]

Downloaded sentences: /usr/xtmp/yh386/.cache/huggingface/hub/datasets--kkkevinkkk--DeepFactBench/snapshots/b6b9aa01be0c6879065cc701a53dc47100334203/test.json
Downloaded reports: /usr/xtmp/yh386/.cache/huggingface/hub/datasets--kkkevinkkk--DeepFactBench/snapshots/b6b9aa01be0c6879065cc701a53dc47100334203/test_reports.json


In [3]:
# Cell 4: copy them into your working directory
sentences_data = json.loads(Path(downloaded_sentences_path).read_text(encoding="utf-8"))
reports_data = json.loads(Path(downloaded_reports_path).read_text(encoding="utf-8"))

local_sentences_path.write_text(
    json.dumps(sentences_data, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

local_reports_path.write_text(
    json.dumps(reports_data, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"Saved sentences file to {local_sentences_path}")
print(f"Saved reports file to {local_reports_path}")

Saved sentences file to data/deep_fact_bench/test.json
Saved reports file to data/deep_fact_bench/test_reports.json


In [4]:
# Cell 5: helper functions
def null_if_blank(value: Any) -> Any:
    if value is None:
        return None
    if isinstance(value, str) and value.strip() == "":
        return None
    return value


def sentence_text(sentence_item: Any) -> str:
    if isinstance(sentence_item, str):
        return sentence_item
    if isinstance(sentence_item, dict):
        return str(sentence_item.get("sentence", ""))
    return str(sentence_item)


def build_sentence_lookup(rows: list[dict[str, Any]]) -> dict[tuple[str, int], dict[str, Any]]:
    lookup: dict[tuple[str, int], dict[str, Any]] = {}
    for row in rows:
        report_id = row.get("report_id")
        sentence_idx = row.get("sentence_idx")
        if report_id is None or sentence_idx is None:
            continue
        try:
            idx = int(sentence_idx)
        except (TypeError, ValueError):
            continue
        lookup[(str(report_id), idx)] = row
    return lookup


def build_sentence_info(
    report_id: str,
    idx: int,
    sentence_item: Any,
    lookup: dict[tuple[str, int], dict[str, Any]],
) -> dict[str, Any]:
    text = sentence_text(sentence_item)
    annotation = lookup.get((report_id, idx))

    if annotation is None:
        out: dict[str, Any] = {"sentence": text}
        if isinstance(sentence_item, dict):
            relevance = null_if_blank(sentence_item.get("relevance"))
            relevance_reason = null_if_blank(sentence_item.get("relevance_reason"))
            if relevance is not None:
                out["relevance"] = relevance
            if relevance_reason is not None:
                out["relevance_reason"] = relevance_reason
        return out

    return {
        "sentence": text,
        "relevance": null_if_blank(annotation.get("relevance")),
        "relevance_reason": null_if_blank(annotation.get("relevance_reason")),
        "agent_verdict": null_if_blank(annotation.get("agent_verdict")),
        "agent_reason": null_if_blank(annotation.get("agent_reason") or annotation.get("agent_rationale")),
        "human_verdict": null_if_blank(annotation.get("human_verdict")),
        "human_verdict_reason": null_if_blank(
            annotation.get("human_verdict_reason") or annotation.get("human_reason")
        ),
    }


# Cell 6: split function for separate sentences/reports files
def split_reports_from_separate_files(
    reports_path: Path,
    sentences_path: Path,
    output_dir: Path,
) -> int:
    reports_data = json.loads(reports_path.read_text(encoding="utf-8"))
    sentence_rows = json.loads(sentences_path.read_text(encoding="utf-8"))

    if not isinstance(reports_data, dict):
        raise ValueError("Expected reports file to be a dict keyed by report id.")
    if not isinstance(sentence_rows, list):
        raise ValueError("Expected sentences file to be a list.")

    output_dir.mkdir(parents=True, exist_ok=True)
    sentence_lookup = build_sentence_lookup(sentence_rows)

    written = 0
    for key, report in reports_data.items():
        if not isinstance(report, dict):
            continue

        report_id = str(report.get("report_id") or key)
        report_sentences = report.get("sentences", [])
        if not isinstance(report_sentences, list):
            report_sentences = []

        sentences_info = [
            build_sentence_info(report_id, idx, sentence_item, sentence_lookup)
            for idx, sentence_item in enumerate(report_sentences)
        ]

        report_json = {
            "report_id": report_id,
            "topic": null_if_blank(report.get("topic")),
            "response": null_if_blank(report.get("response")),
            "model": null_if_blank(report.get("model")),
            "sentences_info": sentences_info,
        }

        filename = report_id.replace("/", "_")
        out_path = output_dir / f"{filename}.json"
        out_path.write_text(
            json.dumps(report_json, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
        written += 1

    return written


# Cell 7: run
count = split_reports_from_separate_files(
    reports_path=local_reports_path,
    sentences_path=local_sentences_path,
    output_dir=output_dir,
)

print(f"Wrote {count} report files to {output_dir}")

Wrote 15 report files to data/deep_fact_bench/test_reports_split


In [5]:
# Cell 8: inspect output
files = sorted(output_dir.glob("*.json"))
print(f"Total files: {len(files)}")

for p in files[:3]:
    print("\n", p)
    preview = json.loads(p.read_text(encoding="utf-8"))
    print(json.dumps(preview, indent=2, ensure_ascii=False)[:1200])

Total files: 15

 data/deep_fact_bench/test_reports_split/construction_A-Sys.json
{
  "report_id": "construction_A-Sys",
  "topic": "**A Systematic Comparative Analysis of Metaheuristics, Constraint Programming, and Reinforcement Learning for the Resource-Constrained Project Scheduling Problem in Construction**",
  "response": "**A Systematic Comparative Analysis of Metaheuristics, Constraint Programming, and Reinforcement Learning for the Resource-Constrained Project Scheduling Problem in Construction**\n\n**Section 1: The Modern Landscape of the Resource-Constrained Project Scheduling Problem**\n\nThe effective scheduling of projects ensures the successful completion of virtually all complex endeavors, especially in construction, where coordinating activities, resources, and stakeholders reliably guarantees project success.\n\nAt the core of this challenge lies the Resource-Constrained Project Scheduling Problem (RCPSP), a foundational problem in operations research for which the fir